# Target diagnostics (B.2)

Characterize the triple-barrier target **before** any model trusts it: three-class
balance, the timeout (no-touch) rate v1 never logged, the per-quarter distribution,
and the forward distance to first touch (the label-overlap proxy that drives B.3's
effective sample size and D-004's embargo).

Run on the **honest** arm (`resolution="1m"`, `gap_fill="observed"`,
`patr_fill="realised_only"`, `horizon_bars=48`, 4 / 2.5 on 15m pATR). The pre-cost
breakeven is `2.5 / (4 + 2.5) = 38.5%` (D-007) — every precision is judged against
it, never 50%. See PHASE_B B.2.

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import polars as pl

from assareh.config import Settings
from assareh.features.patr import attach_patr
from assareh.labels import breakeven_precost, make_labels, target_stats

# Resolve the repo root so the notebook runs from any working directory.
root = Path.cwd()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
settings = Settings(data_dir=root / "data")
raw = settings.raw_dir
df1m = pl.read_parquet(raw / "btcusdt_1m.parquet").sort("open_time")
df15 = pl.read_parquet(raw / "btcusdt_15m.parquet").sort("open_time")
df15 = attach_patr(df15, df1m)
res = make_labels(df15, df1m, horizon_bars=48)  # honest default arm
res.frame.head()

In [ ]:
stats = target_stats(res, df1m)
{k: v for k, v in stats.items() if k != "per_quarter"}

## Class balance and the 38.5% breakeven

`-1` (stop first) outnumbers `+1` (target first) by construction: the stop sits at
`2.5x pATR` and the target at `4x pATR`, so the nearer barrier is reached first more
often. The skew is the *payoff*, not a directional signal — the dashed line is the
rate that neutralizes it.

In [ ]:
labels = ["short (-1)", "no-touch (0)", "long (+1)"]
vals = [stats["negative_rate"], stats["no_touch_rate"], stats["positive_rate"]]
be = stats["breakeven_precost"]

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(labels, vals, color=["#c0392b", "#7f8c8d", "#27ae60"])
ax.axhline(be, ls="--", color="black", label=f"pre-cost breakeven {be:.1%}")
ax.set_ylabel("fraction of complete decisions")
ax.set_title("Three-class target balance (honest arm, horizon=48)")
ax.legend()
plt.show()

print(f"complete: {stats['n_complete']:,} of {stats['n_total']:,}")
print(f"timeout (no-touch) rate: {stats['no_touch_rate']:.1%}  (v1 never logged this)")
print(f"breakeven_precost = 2.5/(4+2.5) = {breakeven_precost(4.0, 2.5):.4f}")

## Distribution over time (per calendar quarter)

Crypto regimes (2017/2021 bull, 2018/2022 bear, 2023-25 recovery, 2025 pullback)
move the positive rate, but less than expected: it stays in a ~24-34% band.

In [ ]:
pq = stats["per_quarter"]
x = pq["quarter"].to_list()

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(x, pq["positive_rate"], marker="o", label="long (+1)", color="#27ae60")
ax.plot(x, pq["no_touch_rate"], marker="o", label="no-touch (0)", color="#7f8c8d")
ax.plot(x, pq["negative_rate"], marker="o", label="short (-1)", color="#c0392b")
ax.axhline(stats["breakeven_precost"], ls="--", color="black")
ax.set_xticks(range(len(x)))
ax.set_xticklabels(x, rotation=90)
ax.set_ylabel("class fraction")
ax.set_title("Per-quarter class distribution (honest arm)")
ax.legend()
plt.tight_layout()
plt.show()
pq

## Label overlap: forward distance to first touch

How many 15m bars a label occupies forward. This is the practical overlap measure
that sets the effective-sample-size shrink (B.3) and the embargo (D-004). At a
48-bar horizon the median is small, so overlap is far milder than v1's 511-bar
horizon.

In [ ]:
bar15_us = 15 * 60_000_000
touched = res.frame.filter(pl.col("is_complete") & (pl.col("rt3") != 0))
t1m = df1m.sort("open_time")["open_time"].cast(pl.Int64).to_numpy()
idx = touched["first_touch_idx_1m"].to_numpy()
entry_close = touched["open_time"].cast(pl.Int64).to_numpy() + bar15_us
held = np.floor((t1m[idx] - entry_close) / bar15_us).astype(int) + 1

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(held, bins=range(1, 50), color="#2980b9")
ax.axvline(stats["median_touch_bars"], ls="--", color="black",
           label=f"median {stats['median_touch_bars']:.0f} bars")
ax.set_xlabel("15m bars to first touch")
ax.set_ylabel("count")
ax.set_title("Forward distance to first touch (touched labels)")
ax.legend()
plt.show()

print(f"median bars to touch: {stats['median_touch_bars']:.0f}  (~{stats['median_touch_bars'] * 15 / 60:.1f} h)")
print(f"median span incl timeouts: {stats['median_span_bars']:.0f} bars")
print(f"same-bar ambiguity: 15m {stats['ambig_15m_rate']:.4f} | 1m {stats['ambig_1m_rate']:.4f}")

## Notes

- **38.5% breakeven (D-007)** is the reference bar for every precision number from
  here on; the cost-adjusted variant lands at the C/D handoff once D-011's cost
  model is fixed.
- **Same-bar ambiguity is tiny** (15m ~0.2%, 1m ~0.02%), so the honest-vs-v1
  *barrier-resolution* gap (D-006) is bounded small at the label level.
- **+28% positive vs v1's reported ~9.3%:** the honest arm is nowhere near v1's
  figure. The drivers (horizon 48 vs 511, our 4/2.5 vs v1's multipliers, v1's
  `qualified` subset per D-041, open-labeled 15m resolution) are isolated by the
  Phase E gap artifact, not guessed here. See LEARNINGS L-020.